<a href="https://colab.research.google.com/github/galipaydin/pes517/blob/main/notebooks/lab04_first_supervised_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# PES 517 · Lab 4 — Your first supervised model
**Artificial Intelligence Applications in Sport · Week 4**

⏱️ About 70 minutes · 🧪 This notebook creates its own data — nothing can fail to download

> **First:** `File → Save a copy in Drive`. Cells marked ✏️ **EDIT ME** are the ones you change.


### What you will produce
A leaderboard of five models predicting the same sport outcome, and a written recommendation about which one you
would actually deploy — which is **not** necessarily the one at the top.

### The idea in one paragraph
Supervised learning means: here are past examples where we know both the inputs and the answer; find the pattern
that maps one to the other; apply it to a new athlete. Linear regression — which you already know — is a machine
learning model by this definition. What changes is not the mathematics but the **goal**: statistics asks *does X
have an effect?*, machine learning asks *will this work on the next athlete?* Different goals, different validation.


---
## Block 1 — Setup


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance
RNG = np.random.default_rng(4)
print('Setup complete.')


---
## Block 2 — The data

A physical test battery on 200 academy and senior athletes. The outcome we want to predict is **RSA decrement
(%)** — how much an athlete slows down across a repeated-sprint protocol. Lower is better.

You would like to predict it, because the RSA protocol itself is exhausting and you cannot run it every month.


In [ ]:
N = 200
d = pd.DataFrame({
    'age':            RNG.integers(16, 33, N),
    'body_mass_kg':   np.round(RNG.normal(75, 8, N), 1),
    'height_cm':      np.round(RNG.normal(180, 7, N), 1),
    'training_age_y': np.round(np.clip(RNG.normal(7, 3.5, N), 0.5, None), 1),
    'cmj_cm':         np.round(np.clip(RNG.normal(41, 6, N), 20, None), 1),
    'sprint10_s':     np.round(np.clip(RNG.normal(1.74, .09, N), 1.4, None), 3),
    'sprint30_s':     np.round(np.clip(RNG.normal(4.18, .21, N), 3.5, None), 3),
    'yoyo_m':         (RNG.integers(8, 32, N) * 40).astype(int),
    'vo2max':         np.round(np.clip(RNG.normal(56, 6, N), 35, None), 1),
    'hr_rec60_bpm':   RNG.integers(18, 48, N),
    'iso_hamstring_N':RNG.integers(180, 420, N),
    'sessions_per_wk':RNG.integers(3, 10, N),
})

# the truth: RSA decrement is driven by aerobic capacity, recovery and speed reserve
d['rsa_decrement_pct'] = np.round(np.clip(
      14.0
    - 0.085 * d.vo2max
    - 0.055 * d.hr_rec60_bpm
    - 0.0018 * d.yoyo_m
    + 2.10  * (d.sprint30_s - 4.18)
    - 0.030 * (d.cmj_cm - 41)
    + RNG.normal(0, 0.85, N), 1.0, None), 2)

FEATURES = [c for c in d.columns if c != 'rsa_decrement_pct']
TARGET   = 'rsa_decrement_pct'

print(f'{N} athletes, {len(FEATURES)} predictors')
print(d[TARGET].describe().round(2).to_string())
d.head()


---
## Block 3 — Train and test

Split first. **Nothing you learn from the test set is allowed to touch the training set.**


In [ ]:
X, y = d[FEATURES], d[TARGET]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=.25, random_state=0)
print(f'training on {len(X_tr)} athletes, testing on {len(X_te)} athletes the model has never seen')


---
## Block 4 — The leaderboard

Five model families, one loop. Each is fitted on the training athletes and scored on the test athletes.

**MAE** = mean absolute error, in percentage points of RSA decrement — *how wrong is a typical prediction?*
**R²** = the share of variation explained. **CV MAE** = 5-fold cross-validated error inside the training set only.


In [ ]:
models = {
    'Linear regression'   : LinearRegression(),
    'Ridge regression'    : Ridge(alpha=1.0),
    'Decision tree'       : DecisionTreeRegressor(max_depth=4, random_state=0),
    'Random forest'       : RandomForestRegressor(n_estimators=400, min_samples_leaf=2, random_state=0),
    'Gradient boosting'   : GradientBoostingRegressor(random_state=0),
}

rows = []
for name, m in models.items():
    m.fit(X_tr, y_tr)
    pred = m.predict(X_te)
    cv = -cross_val_score(m, X_tr, y_tr, cv=KFold(5, shuffle=True, random_state=0),
                          scoring='neg_mean_absolute_error').mean()
    rows.append({'model': name,
                 'test MAE': round(mean_absolute_error(y_te, pred), 3),
                 'test R2' : round(r2_score(y_te, pred), 3),
                 'CV MAE'  : round(cv, 3),
                 'explainable to a coach?': 'yes' if 'regression' in name or 'tree' in name else 'not really'})

board = pd.DataFrame(rows).sort_values('test MAE').reset_index(drop=True)
board


In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
b = board.sort_values('test MAE', ascending=False)
cols = ['#1B6470' if 'regression' in n else '#8A1C2B' for n in b.model]
ax.barh(b.model, b['test MAE'], color=cols)
for i, (v, n) in enumerate(zip(b['test MAE'], b.model)):
    ax.text(v + .01, i, f'{v:.3f}', va='center', fontsize=10)
ax.set_xlabel('Test MAE  (percentage points of RSA decrement — lower is better)')
ax.set_title('Five models, same data'); ax.grid(axis='x', alpha=.3)
plt.tight_layout(); plt.show()


---
## Block 5 — What is the model looking at?

Permutation importance: shuffle one column and see how much worse the model gets. If shuffling a column costs
nothing, the model was not using it.


In [ ]:
best_name = board.loc[0, 'model']
best = models[best_name].fit(X_tr, y_tr)

pi = permutation_importance(best, X_te, y_te, n_repeats=20, random_state=0)
imp = pd.Series(pi.importances_mean, index=FEATURES).sort_values()

fig, ax = plt.subplots(figsize=(8, 4.4))
ax.barh(imp.index, imp.values, color='#1B6470')
ax.set_xlabel('Increase in error when this column is shuffled')
ax.set_title(f'What {best_name} uses'); ax.grid(axis='x', alpha=.3)
plt.tight_layout(); plt.show()


---
### Look at what won

On this dataset **linear regression is at the top of the leaderboard**, and gradient boosting and random forest are
measurably worse. That is not a bug, and it is not unusual in sport science. Three reasons:

1. **The underlying relationship really is close to linear.** Aerobic capacity, recovery and speed reserve each
   push repeated-sprint ability in one direction, roughly proportionally. Tree ensembles spend their capacity
   approximating a straight line out of steps.
2. **n = 200.** Flexible models need data to find the structure that justifies their flexibility. Two hundred
   athletes is a large sport-science dataset and a small machine-learning one.
3. **Twelve clean predictors, no interactions worth finding.** Boosting earns its keep on wide, messy, interacting
   data — GPS traces, event streams, images. Not on a test battery.

> **The generalisable lesson:** *complexity is a cost you pay in interpretability and sample size, and it has to be
> earned by a real improvement in error.* Run the simple model first, always. If the complicated one cannot beat
> it, you have learned something valuable about your problem — and you get to keep the model you can explain.

Check the permutation-importance plot below against physiology. It should be naming the variables a sport
scientist would have named without any model at all. When it does, that is evidence your pipeline is sound.


---
## Block 6 — Break it 💥

Change `TEST_FRACTION` below to `0.99`: the model now trains on two athletes.
Then try `0.01`: it is tested on two. Neither number means anything.


In [ ]:
TEST_FRACTION = 0.99   # ✏️ EDIT ME — try 0.99, then 0.01, then 0.25

Xa, Xb, ya, yb = train_test_split(X, y, test_size=TEST_FRACTION, random_state=0)
m = RandomForestRegressor(n_estimators=200, random_state=0).fit(Xa, ya)
print(f'trained on {len(Xa)} athletes, tested on {len(Xb)}')
print(f'test MAE = {mean_absolute_error(yb, m.predict(Xb)):.3f}   R2 = {r2_score(yb, m.predict(Xb)):.3f}')


---
## Optional — the two-line version

PyCaret fits and cross-validates a dozen models from one call. It is genuinely useful and it is also a slow,
fragile install on the free tier, so it is optional here and everything above already works without it.

Uncomment and run only if you have five minutes to spare.


In [ ]:
# !pip install -q pycaret==3.3.2
# from pycaret.regression import setup, compare_models, plot_model
# s    = setup(d, target='rsa_decrement_pct', session_id=42, train_size=0.75, verbose=False)
# best = compare_models(n_select=1)
# plot_model(best, plot='feature')


---
## Interpretation ✍️ **(graded)**

About 150 words:

1. Which model is at the top of your leaderboard, and by how much — **in percentage points of RSA decrement** —
   does it beat the worst? Is that difference meaningful for the decision this would inform?
2. Linear regression outperformed both tree ensembles here. Give two reasons why, and name one kind of sport
   dataset where you would expect the opposite.
3. The test MAE and the cross-validated MAE differ. Which would you quote to your head of performance, and why?
4. The RSA protocol is exhausting, which is why you wanted to predict it. Given your best model's error of about
   ±0.8 percentage points, would you actually stop running the protocol? Justify.

---

**Your answer:**

*(replace this line)*


---
## AI Use Log

Complete honestly — it is part of the mark. Write *none* if you used no AI tool.

| Tool | What I asked it | What it gave me | What I changed or rejected |
|---|---|---|---|
|  |  |  |  |


---
### Submitting

1. `File → Save a copy in Drive` · 2. `Share → Anyone with the link → Viewer` · 3. Paste the link into ODTÜClass.
